In [55]:

# imports
import json
import pandas as pd
from os import path

# constants
intersection_data_path = ("~/code/county_coverage/data/raw/intersections/" +
                            "Jefferson_County_KY_Street_Intersections.geojson")

In [103]:
def get_records(intersection_data_path):
    with open(idp, 'r') as file:
        data = json.load(file)
    features = data['features']
    for feature in features:
        properties = feature['properties']
        geometry = feature['geometry']
        # properties['geo_type'] = geometry['type'] # Always == "Point". Not useful.
        properties['GEOMETRY'] = geometry['coordinates']
        yield properties

df = pd.DataFrame.from_dict(get_records(path.expanduser(intersection_data_path))).convert_dtypes()
df.head()

,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID,GEOMETRY
0,1,5464,7662,154647662,1,,REHL,RD,W,REHL,CT,1278243.0,259531.9375,4976,6856,{CD0D9D51-41FB-46FF-B229-AC9C0DDB7E61},"[-85.51044384084946, 38.20588660809318]"
1,2,5464,6551,254646551,2,,REHL,RD,,TUCKER STATION,RD,1273126.375,257588.25,4976,5908,{A9FAED81-F7AE-436F-A657-F95FE1B905E8},"[-85.52816882852979, 38.20037561255241]"
2,3,5464,6551,354646551,3,,REHL,RD,,TUCKER STATION,RD,1273050.50875,257590.85375,4976,5908,{71155AF3-3487-4EDA-8B72-B282378DE7F3},"[-85.52841872335158, 38.200360349057064]"
3,4,3194,9996,431949996,4,,I 64 EAST,,,I 265 RAMP,,1279903.25,265597.5,3076,8763,{AD332CAB-27B0-48B7-ACBF-5CDEEC31654B},"[-85.50495414554669, 38.22260344055154]"
4,5,9349,9996,593499996,5,,I 265 NORTH,,,I 265 RAMP,,1279730.75,265426.4375,8197,8763,{0FE38BAB-8B39-4BE8-BA57-1DAA46FDD4BC},"[-85.50554642815675, 38.222127288727606]"


In [ ]:
df1 = df.drop(["GLOBALID"], axis=1)

bad_row = 100354
# see notes below
# full of nulls that mess up road name compression
# remove row before next step

df1 = df1.drop(df1[df1.OBJECTID == bad_row].index)


In [106]:

# Compress road name info
fst_road_info = df1[["FST_INTPRE", "FST_INTNAME", "FST_INTSUF"]].apply(" ".join, axis=1).str.strip()
sec_road_info = df1[["SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"]].apply(" ".join, axis=1).str.strip()

intersections = df1.drop(["FST_INTPRE", "FST_INTNAME", "FST_INTSUF", 
                          "SEC_INTPRE", "SEC_INTNAME", "SEC_INTSUF"], axis=1)
intersections['FST_ROADNAME'] = fst_road_info
intersections['SEC_ROADNAME'] = sec_road_info
intersections.head()


,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GEOMETRY,FST_ROADNAME,SEC_ROADNAME
0,1,5464,7662,154647662,1,1278243.0,259531.9375,4976,6856,"[-85.51044384084946, 38.20588660809318]",REHL RD,W REHL CT
1,2,5464,6551,254646551,2,1273126.375,257588.25,4976,5908,"[-85.52816882852979, 38.20037561255241]",REHL RD,TUCKER STATION RD
2,3,5464,6551,354646551,3,1273050.50875,257590.85375,4976,5908,"[-85.52841872335158, 38.200360349057064]",REHL RD,TUCKER STATION RD
3,4,3194,9996,431949996,4,1279903.25,265597.5,3076,8763,"[-85.50495414554669, 38.22260344055154]",I 64 EAST,I 265 RAMP
4,5,9349,9996,593499996,5,1279730.75,265426.4375,8197,8763,"[-85.50554642815675, 38.222127288727606]",I 265 NORTH,I 265 RAMP


In [102]:
df.INTID.is_unique # True: can use as index?

df.INTID.str.strip().is_unique # still true
set(''.join(df.INTID.str.strip())) == {'0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'C', 'D', 'E', 'F'}
# only hex chars

# INTID is a string but it represents a number.
# derived from SIFCODES 1 and 2

new_intid = df.INTID.str.strip().apply(lambda x:int(x, base=16)).convert_dtypes()
new_intid.is_unique # still true
new_intid

intersections.INTID = new_intid

intersections.head()


,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,FST_ROADNAME,SEC_ROADNAME
OBJECTID,,,,,,,,,,
1,5464,7662,5710837346,1,1.278243e+06,259531.93750,4976,6856,REHL RD,W REHL CT
2,5464,6551,10005800273,2,1.273126e+06,257588.25000,4976,5908,REHL RD,TUCKER STATION RD
3,5464,6551,14300767569,3,1.273051e+06,257590.85375,4976,5908,REHL RD,TUCKER STATION RD
4,3194,9996,18011691414,4,1.279903e+06,265597.50000,3076,8763,I 64 EAST,I 265 RAMP
5,9349,9996,23945910678,5,1.279731e+06,265426.43750,8197,8763,I 265 NORTH,I 265 RAMP


Why are FST_SIFID and SEC_SIFID floats on import?

-> because of a NAN value in item where OBJECTID == 100354
bad_row = 100354

both CSV and JSON are like this

#### CSV:
```csv
X,Y,OBJECTID,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,FST_INTPRE,FST_INTNAME,FST_INTSUF,SEC_INTPRE,SEC_INTNAME,SEC_INTSUF,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,GLOBALID
```

1227948.0,272835.375,100354,,,,,,,,,,,1227948,272835.375,,,{60D18EC3-BBBC-419B-8A10-62C37F39E987}

#### JSON:
```json
null = float('nan')
{ "type": "Feature",
  "properties": {
     "OBJECTID": 100354, "SIFCODE1": null, "SIFCODE2": null, "INTID": null, "SCCAD_ID": null,
     "FST_INTPRE": null, "FST_INTNAME": null, "FST_INTSUF": null, "SEC_INTPRE": null, "SEC_INTNAME": null,
     "SEC_INTSUF": null, "X_COORD": 1227948.0, "Y_COORD": 272835.375, "FST_SIFID": null, "SEC_SIFID": null,
     "GLOBALID": "{60D18EC3-BBBC-419B-8A10-62C37F39E987}" },
      
  "geometry": { "type": "Point", "coordinates": [ -85.686176099576869, 38.240392433657043 ] } }
```

This interferes with some code that simplifies the road names. Remove the row with nulls and any other before further processing. 


In [104]:
#df[['X_COORD', 'Y_COORD']].agg((min, max))

# via https://www.lojic.org/sites/default/files/metadata/address_intersection.htm

#Extent 
#Geographic extent 
#Bounding rectangle 
#Extent type  Extent used for searching
west_longitude = -85.945347
east_longitude = -85.344499
north_latitude = 38.378034
south_latitude = 38.005894
#* Extent contains the resource Yes

#Extent in the item's coordinate system 
west_longitude_co = 1154395.500000
east_longitude_co = 1325086.990000
south_latitude_co = 188677.437500
north_latitude_co = 321629.781250
#* Extent contains the resource Yes

In [105]:
# convert coordinates from LOJIC CRS to (longitude, latitude)
# LOJIC projection: ESRI:102679
# NAD_1983_StatePlane_Kentucky_North_FIPS_1601_Feet

# Standard long, lat: epsg:4326

from pyproj import CRS
from pyproj.transformer import Transformer

KY_grid_CRS = CRS("ESRI:102679")
longlat_CRS = CRS("epsg:4326")

CRS_transformer = Transformer.from_crs(crs_from=KY_grid_CRS, crs_to=longlat_CRS, always_xy=True).transform

#CRS_transformer.transform(1154395.500000, 188677.437500)


In [106]:
# converting grid point to longitude, latitude

#reindex['coordinates'] = 
XYgrid = df.X_COORD.combine(df.Y_COORD, lambda x, y:(x, y))

# have to convert this way because CRS_transformer expects 2 arguments
long_lat_coordinates = df.X_COORD.combine(df.Y_COORD, CRS_transformer)

intersections['GEOMETRY'] = long_lat_coordinates
# could encode this as two columns: longitude and latitude
# might still do it, I will typically use this geometry as a point that gets unpacked
# but it seems annoying to have to access two columns constantly when accessing one 
# and unpacking the value is so easy.
# Mirrors the GEOMETRY column in centerlines as well, which is a list of points
intersections.head()

,SIFCODE1,SIFCODE2,INTID,SCCAD_ID,X_COORD,Y_COORD,FST_SIFID,SEC_SIFID,FST_ROADNAME,SEC_ROADNAME,GEOMETRY
OBJECTID,,,,,,,,,,,
1,5464,7662,5710837346,1,1.278243e+06,259531.93750,4976,6856,REHL RD,W REHL CT,"(-85.51043903006253, 38.205879314617064)"
2,5464,6551,10005800273,2,1.273126e+06,257588.25000,4976,5908,REHL RD,TUCKER STATION RD,"(-85.52814980553048, 38.20034879942594)"
3,5464,6551,14300767569,3,1.273051e+06,257590.85375,4976,5908,REHL RD,TUCKER STATION RD,"(-85.52841390771789, 38.20035305747533)"
4,3194,9996,18011691414,4,1.279903e+06,265597.50000,3076,8763,I 64 EAST,I 265 RAMP,"(-85.5049493351228, 38.22259614360763)"
5,9349,9996,23945910678,5,1.279731e+06,265426.43750,8197,8763,I 265 NORTH,I 265 RAMP,"(-85.50554161759499, 38.22211999190267)"


In [109]:
keep_columns = ["INTID", "FST_ROADNAME", "FST_SIFID",	"SEC_ROADNAME", "SEC_SIFID", "GEOMETRY"]

intersections_clean = intersections[keep_columns].convert_dtypes()
display(intersections_clean.head())


,INTID,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
0,154647662,REHL RD,4976,W REHL CT,6856,"[-85.51044384084946, 38.20588660809318]"
1,254646551,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52816882852979, 38.20037561255241]"
2,354646551,REHL RD,4976,TUCKER STATION RD,5908,"[-85.52841872335158, 38.200360349057064]"
3,431949996,I 64 EAST,3076,I 265 RAMP,8763,"[-85.50495414554669, 38.22260344055154]"
4,593499996,I 265 NORTH,8197,I 265 RAMP,8763,"[-85.50554642815675, 38.222127288727606]"


In [ ]:

# write transformed data to file.
store_path = "/Users/bencampbell/code/county_coverage/data/cleaner/intersections_data.json"

intersections_clean.to_json(store_path)

def read_in_intersections(path_to_intersection_json):
    out = pd.read_json(path_to_intersection_json)
    # fix some things on import
    out = out.set_index("INTID")
    out.GEOMETRY = out.GEOMETRY.apply(tuple)
    return out             

read_in_intersections(store_path).head()

#store_path_csv = "/Users/bencampbell/code/county_coverage/data/cleaner/intersections_data.csv"
#inter



,FST_ROADNAME,FST_SIFID,SEC_ROADNAME,SEC_SIFID,GEOMETRY
INTID,,,,,
5710837346,REHL RD,4976,W REHL CT,6856,"(-85.5104390301, 38.2058793146)"
10005800273,REHL RD,4976,TUCKER STATION RD,5908,"(-85.52814980550001, 38.2003487994)"
14300767569,REHL RD,4976,TUCKER STATION RD,5908,"(-85.5284139077, 38.2003530575)"
18011691414,I 64 EAST,3076,I 265 RAMP,8763,"(-85.5049493351, 38.2225961436)"
23945910678,I 265 NORTH,8197,I 265 RAMP,8763,"(-85.5055416176, 38.2221199919)"
